In [1]:
# Cell 1: Import libraries
import json
import zipfile
import re
import os
import time
import concurrent.futures
import numpy as np
from openai import OpenAI
print('Libraries imported successfully.')

Libraries imported successfully.


In [3]:
# Cell 2: Connect to API and set file paths
client = OpenAI(
    api_key='',
    base_url='',
)

CFR_JSON_PATH = r'medical_device_CFR_PCDV.json'
DATASET_PATH  = r'dataset100.zip'
EMBED_CACHE   = r'cfr_embeddings.json'

print('API client ready. Paths configured.')

API client ready. Paths configured.


In [5]:
# Cell 3: Load CFR JSON and build index
# Structure: Part -> Subpart -> Entry -> Product Code
#
# Each product code stores:
#   text             — string used for embedding (see construction below)
#   full_text        — complete regulatory text, passed to GPT as candidate definition
#   devices          — up to 5 historically registered device names under this code
#   definition       — product-code-specific definition
#   physical_state   — physical form of the device
#   technical_method — method of action or measurement
#   target_area      — anatomical or clinical target area

def load_cfr_index(json_path):
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    index = {}
    for part in data['parts']:
        for subpart in part.get('subparts', []):
            for entry in subpart.get('entries', []):
                for pc in entry.get('product_codes', []):

                    # Collect the four code-specific fields (may be empty strings)
                    definition       = pc.get('definition', '').strip()
                    physical_state   = pc.get('physical_state', '').strip()
                    technical_method = pc.get('technical_method', '').strip()
                    target_area      = pc.get('target_area', '').strip()

                    # Build extra block — only include non-empty fields
                    extra_parts = [s for s in [
                        definition, technical_method, physical_state, target_area
                    ] if s]
                    extra_block = ' '.join(extra_parts)

                    # Embedding text = product title + entry title + FULL regulatory text
                    #                  + code-specific distinguishing fields
                    text = f"{pc['title']}. {entry['title']}. {entry['full_text']}"
                    if extra_block:
                        text = text + ' ' + extra_block

                    index[pc['id']] = {
                        'regulation_id':    entry['id'],
                        'regulation_title': entry['title'],
                        'product_code':     pc['id'],
                        'product_title':    pc['title'],
                        'device_class':     pc['device_class'],
                        'text':             text,
                        'full_text':        entry['full_text'],
                        'devices':          [d['title'] for d in pc.get('devices', [])[:5]],
                        'definition':       definition,
                        'physical_state':   physical_state,
                        'technical_method': technical_method,
                        'target_area':      target_area,
                    }

    print(f'CFR index built: {len(index)} product codes loaded.')

    # Quick sanity check: show text length distribution
    lengths = sorted(len(v['text']) for v in index.values())
    n = len(lengths)
    print(f'Embedding text lengths — min:{lengths[0]}  median:{lengths[n//2]}  '
          f'p90:{lengths[int(n*0.9)]}  max:{lengths[-1]} chars')

    return index

cfr_index = load_cfr_index(CFR_JSON_PATH)


CFR index built: 5955 product codes loaded.
Embedding text lengths — min:213  median:954  p90:3086  max:32331 chars


In [7]:
# Cell 4: Define embedding utilities

def get_embedding(text):
    """Call the OpenAI embeddings API and return a vector for the given text."""
    response = client.embeddings.create(
        model='text-embedding-3-small',
        input=text
    )
    return response.data[0].embedding

def cosine_similarity(vec_a, vec_b):
    """Compute cosine similarity between two vectors. Range: -1 to 1, higher = more similar."""
    a = np.array(vec_a)
    b = np.array(vec_b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

test_vec = get_embedding('blood pressure monitor')
print(f'Embedding test passed. Vector length: {len(test_vec)}')

Embedding test passed. Vector length: 1536


In [9]:
# Cell 5: Build or load CFR embeddings (hash-based cache validation)
#
# Computes an MD5 fingerprint of all product code texts in cfr_index.
# On load, the fingerprint is compared against the value stored in the cache file.
# If they match, the cache is loaded directly.
# If they differ, the stale cache is deleted and embeddings are regenerated.
# No manual intervention required when Cell 3 is updated.

import hashlib

def compute_index_hash(cfr_index):
    h = hashlib.md5()
    for code in sorted(cfr_index.keys()):
        h.update(cfr_index[code]['text'].encode('utf-8'))
    return h.hexdigest()

def get_single_embedding(args):
    code, text = args
    for attempt in range(3):
        try:
            response = client.embeddings.create(
                model='text-embedding-3-small',
                input=text
            )
            return code, response.data[0].embedding
        except Exception as e:
            if attempt == 2:
                print(f'  Failed after 3 attempts: {code} — {e}')
                return code, None
            time.sleep(1)

def build_or_load_embeddings(cfr_index, cache_path, max_workers=20):
    current_hash = compute_index_hash(cfr_index)

    if os.path.exists(cache_path):
        with open(cache_path, 'r') as f:
            cached = json.load(f)

        if cached.get('__hash__') == current_hash:
            embeddings = {k: v for k, v in cached.items() if not k.startswith('__')}
            print(f'Cache valid. Loaded {len(embeddings)} embeddings.')
            return embeddings
        else:
            print('Index hash mismatch. Rebuilding embeddings from scratch...')
            os.remove(cache_path)
    else:
        print('No cache found. Generating embeddings...')

    tasks = [(code, info['text']) for code, info in cfr_index.items()]
    embeddings = {}

    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(get_single_embedding, task): task for task in tasks}
        done = 0
        for future in concurrent.futures.as_completed(futures):
            code, vec = future.result()
            if vec is not None:
                embeddings[code] = vec
            done += 1
            if done % 500 == 0:
                print(f'  Progress: {done}/{len(tasks)}')

    to_save = dict(embeddings)
    to_save['__hash__'] = current_hash
    with open(cache_path, 'w') as f:
        json.dump(to_save, f)
    print(f'Done. {len(embeddings)} embeddings saved to: {cache_path}')
    return embeddings

cfr_embeddings = build_or_load_embeddings(cfr_index, EMBED_CACHE, max_workers=20)

Cache valid. Loaded 5955 embeddings.


In [10]:
missing = [code for code in cfr_index if code not in cfr_embeddings]
print(f'Number of missing items: {len(missing)}')
print(f'Missing code: {missing}')

Number of missing items: 0
Missing code: []


In [13]:
for code in missing:
    text = cfr_index[code]['text']
    if len(text) > 6000:
        text = text[:6000]
    try:
        response = client.embeddings.create(
            model='text-embedding-3-small',
            input=text
        )
        cfr_embeddings[code] = response.data[0].embedding
        print(f'Patched: {code}')
    except Exception as e:
        print(f'Failed: {code} — {e}')

import hashlib

def compute_index_hash(cfr_index):
    h = hashlib.md5()
    for code in sorted(cfr_index.keys()):
        h.update(cfr_index[code]['text'].encode('utf-8'))
    return h.hexdigest()

to_save = dict(cfr_embeddings)
to_save['__hash__'] = compute_index_hash(cfr_index)
with open(EMBED_CACHE, 'w') as f:
    json.dump(to_save, f)
print('Cache updated.')

print(f'Total embeddings: {len(cfr_embeddings)}')  # should be 5955

Cache updated.
Total embeddings: 5955


In [14]:
print(len(cfr_embeddings))  # should be 5955

5955


In [17]:
# Cell 6: Load and parse a device file from the dataset
# Path A: device_description is not null -> use directly
# Path B: device_description is null     -> extract from summary_raw
# Ground truth is always extracted from summary_raw via regex.

def extract_description_from_summary(summary):
    desc_match = re.search(
        r'(The [A-Z][^.]{10,300}(?:intended|designed|used for)[^.]{10,300}\.)',
        summary
    )
    if desc_match:
        return desc_match.group(1).strip(), 'extracted_from_summary'
    name_match = re.search(r'Trade/Device Name:\s*(.+)', summary)
    if name_match:
        return name_match.group(1).strip(), 'device_name_only'
    return None, 'no_description_found'

def load_device(zip_path, filename):
    with zipfile.ZipFile(zip_path) as z:
        with z.open(filename) as f:
            data = json.load(f)

    description  = data.get('device_description')
    intended_use = data.get('intended_use')
    summary      = data.get('summary_raw', '')

    if description:
        if intended_use:
            description = description + ' ' + intended_use
        path_label = 'A_non_null'
        extraction_method = 'direct'
    else:
        description, extraction_method = extract_description_from_summary(summary)
        path_label = 'B_null'

    reg_match  = re.search(r'Regulation Number:\s*(21 CFR [\d\.]+)', summary)
    code_match = re.search(r'Product Code:\s*([A-Z]{2,4})', summary)

    return {
        'k_number':          filename.replace('.json', ''),
        'description':       description,
        'true_regulation':   reg_match.group(1).strip()  if reg_match  else 'UNKNOWN',
        'true_code':         code_match.group(1).strip() if code_match else 'UNKNOWN',
        'path':              path_label,
        'extraction_method': extraction_method,
    }

print('load_device function defined.')

load_device function defined.


In [19]:
# Cell 7: Split ALL 100 files into Path A and Path B

with zipfile.ZipFile(DATASET_PATH) as z:
    all_files = z.namelist()

print(f'Total files in dataset: {len(all_files)}')

path_a_files = []
path_b_files = []

for filename in all_files:
    device = load_device(DATASET_PATH, filename)
    if device['path'] == 'A_non_null':
        path_a_files.append(filename)
    else:
        path_b_files.append(filename)

print(f'Path A (non-null description): {len(path_a_files)} files')
print(f'Path B (null description):     {len(path_b_files)} files')
print(f'Total: {len(path_a_files) + len(path_b_files)} files — running evaluation on all of them.')

Total files in dataset: 100
Path A (non-null description): 97 files
Path B (null description):     3 files
Total: 100 files — running evaluation on all of them.


In [77]:
# Cell 8: Step 1 — Embedding retrieval with query rewriting
# Before embedding, GPT rewrites the commercial description into FDA regulatory language.
# This reduces the semantic gap between device descriptions and CFR regulatory text.

def rewrite_query(client, device_description):
    """Use GPT to rewrite a commercial device description into FDA regulatory language."""
    response = client.chat.completions.create(
        model='gpt-4.1',
        messages=[
            {
                'role': 'system',
                'content': (
                    'You are an FDA regulatory expert. '
                    'Rewrite the following medical device description using precise FDA regulatory terminology. '
                    'Focus on: the device type, intended use, anatomical location, mechanism of action, and regulatory category. '
                    'Be concise (2-3 sentences). Use the kind of language found in 21 CFR regulations.'
                )
            },
            {
                'role': 'user',
                'content': f'Rewrite this description in FDA regulatory language:\n\n{device_description}'
            }
        ],
        temperature=0,
        max_tokens=200,
    )
    return response.choices[0].message.content.strip()

def retrieve_top_candidates(query_text, cfr_embeddings, cfr_index, top_n=40, use_rewriting=True):
    """
    Return the top_n most semantically similar product codes for the given query.
    If use_rewriting=True, GPT first rewrites the query into FDA regulatory language
    before embedding, reducing the semantic gap between commercial and regulatory text.
    """
    if use_rewriting and query_text:
        rewritten = rewrite_query(client, query_text)
        # Combine original + rewritten for richer embedding signal
        embed_text = query_text + ' ' + rewritten
    else:
        embed_text = query_text

    query_vec = get_embedding(embed_text)

    scores = []
    for code, vec in cfr_embeddings.items():
        sim = cosine_similarity(query_vec, vec)
        scores.append((code, sim))

    scores.sort(key=lambda x: x[1], reverse=True)

    candidates = []
    for code, sim in scores[:top_n]:
        info = cfr_index[code]
        candidates.append({
            'product_code':     code,
            'product_title':    info['product_title'],
            'regulation_id':    info['regulation_id'],
            'regulation_title': info['regulation_title'],
            'device_class':     info['device_class'],
            'similarity':       round(sim, 4),
        })

    return candidates

print('Candidate retrieval function ready. Default top_n=40, use_rewriting=True.')

Candidate retrieval function ready. Default top_n=40, use_rewriting=True.


In [23]:
# Cell 9: Install pydantic (run once)
!pip install pydantic

In [81]:
# Cell 10: Step 2 — GPT classification with Structured Outputs (TWO-STEP: regulation -> code)
# Passes the top 40 candidates to GPT-4.1 for final selection.
#
# Each candidate includes:
#   full_text        — complete regulatory definition (primary basis for GPT decision)
#   definition       — product-code-specific definition (key for distinguishing codes
#                      that share the same entry full_text, e.g. LLZ vs QIH)
#   physical_state   — physical form, helps distinguish lab/chemistry device sub-codes
#   technical_method — method of action, further differentiates near-identical codes
#   target_area      — anatomical target, helps when multiple codes cover similar devices
#   devices          — up to 3 historical examples (supporting evidence only)
# reasoning is placed FIRST in the schema to force the model to think before deciding.
from pydantic import BaseModel
from collections import OrderedDict

class ClassificationResult(BaseModel):
    reasoning: str
    selected_regulation_number: str
    product_code: str
    confidence: float


def build_grouped_candidate_text(candidates, cfr_index):
    """
    Group flat top-N candidates by regulation_number (entry), preserving the
    order in which each regulation first appears (i.e. highest-similarity
    product code within that regulation appears first). Each group is shown
    once with its regulation-level info, followed by ALL product codes that
    belong to that regulation among the candidates (not just the one that
    triggered retrieval) so GPT can see the real sibling codes to disambiguate.
    """
    groups = OrderedDict()
    for c in candidates:
        reg_id = c['regulation_id']
        if reg_id not in groups:
            groups[reg_id] = {
                'regulation_title': c['regulation_title'],
                'device_class':     c['device_class'],
                'codes':            [],
            }
        groups[reg_id]['codes'].append(c['product_code'])

    candidate_text = ''
    for g_i, (reg_id, g) in enumerate(groups.items()):
        candidate_text += (
            f"REGULATION {g_i+1}: 21 CFR {reg_id} — {g['regulation_title']} "
            f"(Class {g['device_class']})\n"
        )
        for code in g['codes']:
            info              = cfr_index[code]
            full_definition   = info.get('full_text', '')
            device_examples   = info.get('devices', [])
            examples_text     = ', '.join(device_examples[:3]) if device_examples else 'No examples available'
            definition        = info.get('definition', '')
            physical_state    = info.get('physical_state', '')
            technical_method  = info.get('technical_method', '')
            target_area       = info.get('target_area', '')

            extra_lines = ''
            if definition:
                extra_lines += f'        Specific definition: {definition}\n'
            if physical_state:
                extra_lines += f'        Physical state: {physical_state}\n'
            if technical_method:
                extra_lines += f'        Technical method: {technical_method}\n'
            if target_area:
                extra_lines += f'        Target area: {target_area}\n'

            candidate_text += (
                f"    Product Code: {code} — {info['product_title']}\n"
                f"        Definition: {full_definition}\n"
                f"{extra_lines}"
                f"        Examples of previously registered devices: {examples_text}\n"
            )
        candidate_text += '\n'

    return candidate_text


def classify_with_gpt(client, device_description, candidates, cfr_index):

    candidate_text = build_grouped_candidate_text(candidates, cfr_index)

    system_prompt = """You are an expert FDA medical device regulatory classifier.

You will be given:
1. A medical device description
2. A shortlist of candidate FDA regulations, pre-filtered by semantic similarity. \
Each regulation may list ONE OR MORE product codes underneath it.

Your task has two steps, in order:
STEP 1 — Select the single best matching REGULATION (selected_regulation_number) by \
comparing the device against each regulation's scope and the product codes it contains.
STEP 2 — Within that chosen regulation only, select the single best matching \
PRODUCT CODE (product_code) from the codes listed under it.

Rules:
- You MUST always select a regulation and product code from the candidate list provided.
- The product_code you return MUST be one of the codes listed under the \
selected_regulation_number you return. Never mix a code from a different regulation group.
- Never return an empty product_code, an empty selected_regulation_number, or a \
code that is not in the candidate list.
- Do not assume earlier candidates are more correct. Evaluate all candidates independently.
- Prioritize the full regulatory definition over product title or examples.
- Use examples only as supporting evidence, not as the primary basis for classification.
- In the reasoning field, first identify the key characteristics of the device, then \
compare them with the most relevant candidate regulations, then justify the specific \
product code chosen within the selected regulation.
- Explain why the selected regulation and product code fit better than the closest alternatives.
- If the description is insufficient or ambiguous, still select the closest match, but \
set confidence below 0.5.
- Examples of previously registered devices are provided as reference only.
  Do not select a product code solely because its examples resemble the device.
  The regulatory definition must be the primary basis for selection.

Confidence rules:
- Use confidence above 0.8 only when the device clearly matches the selected definition.
- Use confidence between 0.5 and 0.8 when the match is plausible but alternatives are also possible.
- Use confidence below 0.5 when the description is insufficient, ambiguous, or weakly matched.
"""

    user_prompt = f"""Classify the following medical device.

Step 1: First analyze the device characteristics and compare them with the candidate
regulations listed below. Select the single best matching regulation.
Step 2: Then, within that regulation's listed product codes only, select the single
best matching product code.

DEVICE DESCRIPTION:
{device_description}

CANDIDATE REGULATIONS AND THEIR PRODUCT CODES:
{candidate_text}
"""

    response = client.beta.chat.completions.parse(
        model='gpt-4.1',
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user',   'content': user_prompt},
        ],
        response_format=ClassificationResult,
        temperature=0,
    )

    result = response.choices[0].message.parsed

    # --- Validation: does product_code actually belong to selected_regulation_number? ---
    actual_regulation = None
    code_is_valid = result.product_code in cfr_index
    if code_is_valid:
        actual_regulation = cfr_index[result.product_code]['regulation_id']

    # Normalize both sides for comparison (strip any "21 CFR " prefix / whitespace)
    def _normalize_reg(s):
        return s.replace('21 CFR', '').strip()

    is_consistent = (
        code_is_valid
        and actual_regulation is not None
        and _normalize_reg(actual_regulation) == _normalize_reg(result.selected_regulation_number)
    )

    if not code_is_valid:
        final_regulation_number = result.selected_regulation_number
    elif is_consistent:
        final_regulation_number = actual_regulation
    else:
        final_regulation_number = actual_regulation

    return {
        'reasoning':                  result.reasoning,
        'product_code':               result.product_code,
        'selected_regulation_number': result.selected_regulation_number,  
        'regulation_number':          final_regulation_number,           
        'confidence':                 result.confidence,
        'regulation_code_consistent': is_consistent,                      
    }

print('Classification function ready: two-step regulation -> product_code selection, with consistency validation.')

Classification function ready: two-step regulation -> product_code selection, with consistency validation.


In [83]:
# Cell 11: Combine both steps into one function
# hybrid_classify = retrieve_top_candidates (top 40) + classify_with_gpt

def hybrid_classify(client, device_description, cfr_embeddings, cfr_index, top_n=40):
    if not device_description:
        return {
            'product_code':               'NO_DESCRIPTION',
            'regulation_number':          'NO_DESCRIPTION',
            'selected_regulation_number': 'NO_DESCRIPTION',
            'confidence':                  0.0,
            'reasoning':                  'No description available for this device.',
            'regulation_code_consistent': None,
        }, []

    candidates = retrieve_top_candidates(device_description, cfr_embeddings, cfr_index, top_n)
    result     = classify_with_gpt(client, device_description, candidates, cfr_index)
    return result, candidates

print('hybrid_classify function defined. Default top_n = 40.')

hybrid_classify function defined. Default top_n = 40.


In [35]:
# Cell 12: Quick single-device test
# Runs the full pipeline on one file to verify everything works before batch evaluation.

sample = load_device(DATASET_PATH, all_files[0])
print(f'K-number:     {sample["k_number"]}')
print(f'Path:         {sample["path"]}')
print(f'Description:  {sample["description"][:120]}...')
print(f'Ground truth: {sample["true_code"]} / {sample["true_regulation"]}')
print('\nRunning hybrid classification...')

result, candidates = hybrid_classify(client, sample['description'], cfr_embeddings, cfr_index, top_n=40)

print(f'\nPrediction:')
print(f'  Selected regulation (step 1): {result["selected_regulation_number"]}')
print(f'  Product code (step 2):        {result["product_code"]}')
print(f'  Final regulation (index-corrected): {result["regulation_number"]}')
print(f'  Confidence:                   {result["confidence"]}')
print(f'  Step1/Step2 consistent:       {result["regulation_code_consistent"]}')
print(f'  Reasoning:                    {result["reasoning"]}')
print(f'\nCorrect: {result["product_code"] == sample["true_code"]}')

candidate_codes = [c['product_code'] for c in candidates]
print(f'True code in top-40 candidates: {sample["true_code"] in candidate_codes}')

K-number:     K252586
Path:         A_non_null
Description:  CADDIE is cloud based artificial intelligence medical device software. CADDIE interfaces with the video feed generated b...
Ground truth: QNP / 21 CFR 876.1520

Running hybrid classification...

Prediction:
  Selected regulation (step 1): 21 CFR 876.1520
  Product code (step 2):        QNP
  Final regulation (index-corrected): 876.1520
  Confidence:                   0.95
  Step1/Step2 consistent:       True
  Reasoning:                    Key characteristics of the device: CADDIE is a cloud-based AI software that interfaces with endoscopic video feeds during colonoscopy, analyzes real-time video, and highlights regions with visual characteristics consistent with colonic mucosal lesions (such as polyps and adenomas). It is intended to assist gastroenterologists in detecting suspected colorectal polyps, but does not make diagnostic decisions or recommendations. It is limited to standard white-light endoscopy imaging and is not

In [37]:
# Cell 13: Batch evaluation — run on ALL 100 files, split by path
# For each device:
#   - runs hybrid_classify (top 40 candidates)
#   - compares predicted code vs true code
#   - records whether true code appeared in top-40 candidates (to identify retrieval failures)
#   - records regulation_code_consistent: whether GPT's step-1 regulation pick and step-2 product_code pick actually agreed (see Cell 10). 
# Reports accuracy separately for Path A, Path B, and overall.

def run_evaluation(file_list, dataset_path, client, cfr_embeddings, cfr_index, label, top_n=40):
    correct = 0
    results = []

    print(f'--- Evaluating {label} ({len(file_list)} files) ---')

    for i, filename in enumerate(file_list):
        device = load_device(dataset_path, filename)

        try:
            pred, candidates = hybrid_classify(client, device['description'], cfr_embeddings, cfr_index, top_n=top_n)
        except Exception as e:
            print(f"{i+1:3}. {device['k_number']} | SKIPPED due to error: {str(e)[:80]}")
            results.append({
                'k_number':                    device['k_number'],
                'path':                        device['path'],
                'extraction_method':           device['extraction_method'],
                'description_preview':         (device['description'] or '')[:100],
                'true_code':                   device['true_code'],
                'true_regulation':             device['true_regulation'],
                'predicted_code':              'ERROR',
                'predicted_regulation':        'ERROR',
                'selected_regulation_number':  'ERROR',
                'confidence':                  0.0,
                'correct':                     False,
                'true_in_top40':               False,
                'regulation_code_consistent':  None,
                'reasoning':                   f'Skipped due to API error: {str(e)[:200]}',
            })
            continue

        is_correct = (pred['product_code'] == device['true_code'])
        if is_correct:
            correct += 1

        candidate_codes = [c['product_code'] for c in candidates]
        in_top40 = device['true_code'] in candidate_codes

        consistency_flag = 'OK' if pred['regulation_code_consistent'] else 'MISMATCH'
        print(
            f"{i+1:3}. {device['k_number']} | "
            f"Predicted: {pred['product_code']:6} | "
            f"True: {device['true_code']:6} | "
            f"{'CORRECT' if is_correct else 'WRONG  '} | "
            f"True in top-40: {in_top40} | "
            f"Reg/Code consistency: {consistency_flag}"
        )

        results.append({
            'k_number':                    device['k_number'],
            'path':                        device['path'],
            'extraction_method':           device['extraction_method'],
            'description_preview':         (device['description'] or '')[:100],
            'true_code':                   device['true_code'],
            'true_regulation':             device['true_regulation'],
            'predicted_code':              pred['product_code'],
            'predicted_regulation':        pred['regulation_number'],
            'selected_regulation_number':  pred['selected_regulation_number'],
            'confidence':                  pred['confidence'],
            'correct':                     is_correct,
            'true_in_top40':               in_top40,
            'regulation_code_consistent':  pred['regulation_code_consistent'],
            'reasoning':                   pred['reasoning'],
        })

    accuracy = correct / len(file_list) * 100 if file_list else 0
    print(f'\n{label} accuracy: {correct}/{len(file_list)} = {accuracy:.1f}%\n')
    return results, accuracy


# Run on ALL files
results_a, acc_a = run_evaluation(path_a_files, DATASET_PATH, client, cfr_embeddings, cfr_index, 'Path A (non-null)', top_n=40)
results_b, acc_b = run_evaluation(path_b_files, DATASET_PATH, client, cfr_embeddings, cfr_index, 'Path B (null)',     top_n=40)

all_results   = results_a + results_b
total_correct = sum(1 for r in all_results if r['correct'])
total_mismatch = sum(1 for r in all_results if r['regulation_code_consistent'] is False)

print(f'=== Overall Summary ===')
print(f'Path A accuracy: {acc_a:.1f}%  ({len(path_a_files)} files)')
print(f'Path B accuracy: {acc_b:.1f}%  ({len(path_b_files)} files)')
print(f'Total accuracy:  {total_correct}/{len(all_results)} = {total_correct/len(all_results)*100:.1f}%')
print(f'Regulation/code internal mismatches (GPT step-1 vs step-2 disagreed): {total_mismatch}/{len(all_results)}')

import json
with open('results.json', 'w', encoding='utf-8') as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2)
print('Saved_results.json')

--- Evaluating Path A (non-null) (97 files) ---
  1. K252586 | Predicted: QNP    | True: QNP    | CORRECT | True in top-40: True | Reg/Code consistency: OK
  2. K252593 | Predicted: MXD    | True: MXD    | CORRECT | True in top-40: True | Reg/Code consistency: OK
  3. K252580 | Predicted: LKM    | True: LKM    | CORRECT | True in top-40: True | Reg/Code consistency: OK
  4. K252576 | Predicted: OUH    | True: GEI    | WRONG   | True in top-40: False | Reg/Code consistency: OK
  5. K252562 | Predicted: OLO    | True: OLO    | CORRECT | True in top-40: True | Reg/Code consistency: OK
  6. K252554 | Predicted: NFT    | True: NFT    | CORRECT | True in top-40: True | Reg/Code consistency: OK
  7. K252540 | Predicted: MSS    | True: MSS    | CORRECT | True in top-40: True | Reg/Code consistency: OK
  8. K252539 | Predicted: QIH    | True: QIH    | CORRECT | True in top-40: True | Reg/Code consistency: OK
  9. K252528 | Predicted: QZB    | True: EOQ    | WRONG   | True in top-40: False | Reg

In [87]:
# Compute funnel accuracy from saved results: top-40 retrieval hit rate,
# regulation accuracy among top-40 hits, code accuracy among regulation-correct cases, and overall Path A code accuracy.
import json
if 'all_results' not in dir():
    with open('results.json', 'r', encoding='utf-8') as f:
        all_results = json.load(f)
    print(f'Loaded {len(all_results)} results from file')

def norm_reg(s):
    return (s or '').replace('21 CFR', '').strip()

n_top40   = sum(1 for r in all_results if r['true_in_top40'])
n_reg_ok  = sum(1 for r in all_results if r['true_in_top40'] and norm_reg(r['predicted_regulation']) == norm_reg(r['true_regulation']))
n_code_ok = sum(1 for r in all_results if r['true_in_top40'] and norm_reg(r['predicted_regulation']) == norm_reg(r['true_regulation']) and r['predicted_code'] == r['true_code'])

n_pathA = sum(1 for r in all_results if r['path'] == 'A_non_null')
n_pathA_code_correct = sum(1 for r in all_results if r['path'] == 'A_non_null' and r['correct'])

print(f'Top-40 hits: {n_top40}/{len(all_results)} = {n_top40/len(all_results)*100:.1f}%')
print(f'Regulation correct (within top-40 hits): {n_reg_ok}/{n_top40} = {n_reg_ok/n_top40*100:.1f}%')
print(f'Code correct (within regulation-correct): {n_code_ok}/{n_reg_ok} = {n_code_ok/n_reg_ok*100:.1f}%')
print(f'Path A overall code accuracy: {n_pathA_code_correct}/{n_pathA} = {n_pathA_code_correct/n_pathA*100:.1f}%')

Top-40 hits: 83/100 = 83.0%
Regulation correct (within top-40 hits): 73/83 = 88.0%
Code correct (within regulation-correct): 61/73 = 83.6%
Path A overall code accuracy: 61/97 = 62.9%


In [65]:
# Cell 14: Analysis — inspect all failed cases in detail
# Shows description, predicted vs true code, whether true code was in top-40,
# whether GPT's step-1 regulation and step-2 product code were internally
# consistent, and GPT's full reasoning for each wrong prediction.
# Covers both Path A and Path B failures.
import json
if 'all_results' not in dir():
    with open('results.json', 'r', encoding='utf-8') as f:
        all_results = json.load(f)
    print(f'Loaded {len(all_results)} results from file')

retrieval_failures = 0
gpt_failures       = 0
mismatch_failures  = 0  # wrong AND GPT's own regulation/code picks disagreed

print('=== FAILED CASES ===')
print()

for r in all_results:
    if not r['correct']:
        failure_type = 'RETRIEVAL FAILURE' if not r['true_in_top40'] else 'GPT SELECTION ERROR'
        if not r['true_in_top40']:
            retrieval_failures += 1
        else:
            gpt_failures += 1
        if r.get('regulation_code_consistent') is False:
            mismatch_failures += 1
            failure_type += ' (+ INTERNAL MISMATCH)'

        print(f"=== {r['k_number']} | Path: {r['path']} | {failure_type} ===")
        print(f"Description:    {r['description_preview']}")
        print(f"Predicted:      {r['predicted_code']} | True: {r['true_code']}")
        print(f"Step1 regulation pick: {r.get('selected_regulation_number')} | Final (index-corrected): {r['predicted_regulation']} | True: {r['true_regulation']}")
        print(f"True in top-40: {r['true_in_top40']}")
        print(f"Confidence:     {r['confidence']}")
        print(f"Reasoning:      {r['reasoning']}")
        print()

total_wrong = len([r for r in all_results if not r['correct']])
total_mismatch_all = len([r for r in all_results if r.get('regulation_code_consistent') is False])
print(f'=== Failure Summary ===')
print(f'Total wrong:                  {total_wrong}')
print(f'Retrieval failures:           {retrieval_failures}  (true code not in top-40 — embedding problem)')
print(f'GPT failures:                 {gpt_failures}  (true code in top-40 but GPT selected wrong — prompt problem)')
print(f'  ...of which internal mismatch: {mismatch_failures}  (GPT\'s step-1 regulation and step-2 code disagreed)')
print(f'Internal mismatches (all cases, correct or not): {total_mismatch_all}/{len(all_results)}')

=== FAILED CASES ===

=== K252576 | Path: A_non_null | RETRIEVAL FAILURE ===
Description:    The Legend X Platform is a software-controlled capital equipment platform that enables the applicati
Predicted:      OUH | True: GEI
Step1 regulation pick: 21 CFR 878.4400 | Final (index-corrected): 878.4400 | True: 21 CFR 878.4400
True in top-40: False
Confidence:     0.9
Reasoning:      Key characteristics of the Legend X Platform: (1) It is a software-controlled capital equipment platform for dermatologic and general surgical procedures; (2) It delivers radiofrequency (RF) energy for ablation/resurfacing (VoluDerm), noninvasive treatment of wrinkles/rhytides (TriPollar RF), and electrical stimulation for muscle conditioning/aesthetic purposes (DMSt); (3) It includes a console, applicators, disposable tips, and software; (4) It is intended for ablation/resurfacing of skin, noninvasive wrinkle treatment, and muscle stimulation for aesthetic purposes. 

Comparing candidate regulations:
- Regula

In [91]:
# Cell 15Export results to Excel
# Requires: pip install openpyxl
import openpyxl
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

wb = openpyxl.Workbook()
ws = wb.active
ws.title = 'Classification Results'

# Column definitions
columns = [
    ('K Number',                  'k_number'),
    ('Path',                      'path'),
    ('Device Description',        'description_preview'),
    ('True Code',                 'true_code'),
    ('True Regulation',           'true_regulation'),
    ('Predicted Code',            'predicted_code'),
    ('Predicted Regulation',      'predicted_regulation'),
    ('GPT Step1 Regulation Pick', 'selected_regulation_number'),
    ('Reg/Code Consistent',       'regulation_code_consistent'),
    ('Correct',                   'correct'),
    ('True in Top-40',            'true_in_top40'),
    ('Failure Type',              None),
    ('Confidence',                'confidence'),
    ('Reasoning',                 'reasoning'),
]

# Header style
header_fill = PatternFill(start_color='1F3864', end_color='1F3864', fill_type='solid')
header_font = Font(bold=True, color='FFFFFF', name='Arial', size=11)

for col_idx, (header, _) in enumerate(columns, 1):
    cell = ws.cell(row=1, column=col_idx, value=header)
    cell.fill = header_fill
    cell.font = header_font
    cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)

# Fill data rows
green_fill  = PatternFill(start_color='E2EFDA', end_color='E2EFDA', fill_type='solid')
red_fill    = PatternFill(start_color='FCE4D6', end_color='FCE4D6', fill_type='solid')
orange_fill = PatternFill(start_color='FFF2CC', end_color='FFF2CC', fill_type='solid')
skip_fill   = PatternFill(start_color='D9D9D9', end_color='D9D9D9', fill_type='solid')

for row_idx, r in enumerate(all_results, 2):
    # Determine failure type
    if r['predicted_code'] == 'ERROR':
        failure_type = 'API Skipped'
        row_fill = skip_fill
    elif r['correct']:
        failure_type = '—'
        row_fill = green_fill
    elif not r['true_in_top40']:
        failure_type = 'Retrieval Failure'
        row_fill = red_fill
    else:
        failure_type = 'GPT Selection Error'
        row_fill = orange_fill

    consistent_val = r.get('regulation_code_consistent')
    if consistent_val is None:
        consistent_display = 'N/A'
    elif consistent_val:
        consistent_display = 'YES'
    else:
        consistent_display = 'NO'

    values = {
        'k_number':                    r['k_number'],
        'path':                        r['path'],
        'description_preview':         r['description_preview'],
        'true_code':                   r['true_code'],
        'true_regulation':             r['true_regulation'],
        'predicted_code':              r['predicted_code'],
        'predicted_regulation':        r['predicted_regulation'],
        'selected_regulation_number':  r.get('selected_regulation_number', ''),
        'regulation_code_consistent':  consistent_display,
        'correct':                     'YES' if r['correct'] else 'NO',
        'true_in_top40':               'YES' if r['true_in_top40'] else 'NO',
        'confidence':                  r['confidence'],
        'reasoning':                   r['reasoning'],
    }

    for col_idx, (_, field) in enumerate(columns, 1):
        val = failure_type if field is None else values.get(field, '')
        cell = ws.cell(row=row_idx, column=col_idx, value=val)
        cell.fill = row_fill
        cell.font = Font(name='Arial', size=10)
        cell.alignment = Alignment(vertical='top', wrap_text=True)
        # Highlight internal mismatches regardless of row fill, in their own column
        if field == 'regulation_code_consistent' and consistent_val is False:
            cell.font = Font(name='Arial', size=10, bold=True, color='C00000')

# Column widths
col_widths = [12, 8, 50, 10, 20, 12, 20, 20, 14, 8, 12, 18, 10, 80]
for i, width in enumerate(col_widths, 1):
    ws.column_dimensions[get_column_letter(i)].width = width

# Row heights
ws.row_dimensions[1].height = 30
for row_idx in range(2, len(all_results) + 2):
    ws.row_dimensions[row_idx].height = 60

# Freeze header row
ws.freeze_panes = 'A2'

wb.save('classification_results.xlsx')
print(f'Done. {len(all_results)} rows exported to classification_results.xlsx')

Done. 100 rows exported to classification_results.xlsx
